from pickle import load
from os import mkdir
from cv2 import imwrite

labels = [
    i.decode()
    for i in load(open("../dat/cifar-100-python/meta", "rb"), encoding="bytes")[
        b"fine_label_names"
    ]
]
for i in labels:
    mkdir("../dat/cifar100/" + i)
for i in "train", "test":
    a = load(open(f"../dat/cifar-100-python/{i}", "rb"), encoding="bytes")
    for i, j, k in zip(a[b"fine_labels"], a[b"filenames"], a[b"data"]):
        imwrite(
            f"../dat/cifar100/{labels[i]}/{j.decode()}",
            k.reshape(3, 32, 32).transpose(1, 2, 0),
        )
labels = [
    i.decode()
    for i in load(
        open("../dat/cifar-10-batches-py/batches.meta", "rb"), encoding="bytes"
    )[b"label_names"]
]
for i in labels:
    mkdir("../dat/cifar10/" + i)
for i in (
    "data_batch_1",
    "data_batch_2",
    "data_batch_3",
    "data_batch_4",
    "data_batch_5",
    "test_batch",
):
    a = load(open(f"../dat/cifar-10-batches-py/{i}", "rb"), encoding="bytes")
    for i, j, k in zip(a[b"labels"], a[b"filenames"], a[b"data"]):
        imwrite(
            f"../dat/cifar10/{labels[i]}/{j.decode()}",
            k.reshape(3, 32, 32).transpose(1, 2, 0),
        )

In [ ]:
from torch.nn import *
import torch
import numpy as np


class IterPool:
    def __init__(s, f, a, g=Identity(), b=1, p=3):
        s.f = f
        s.a = a
        s.g = g
        s.b = b
        s.p = []
        for _ in range(b * p):
            s.fill()

    def __iter__(s):
        return s

    def __next__(s):
        r = []
        while len(r) < s.b:
            i = np.random.randint(len(s.p))
            try:
                r.append(next(s.p[i]))
            except StopIteration:
                del s.p[i]
                s.fill()
        return s.g(r)

    def fill(s):
        s.p.append(iter(s.f(s.a[np.random.randint(len(s.a))])))

from torchvision.transforms import *
from random import choice, sample
from PIL.Image import open
from glob import glob
import torchvision


class Cutout:
    def __init__(s, r):
        s.r = r

    def __call__(s, a):
        a = a.cuda()
        h, w = a.shape[1:]
        x = np.random.randint(h)
        y = np.random.randint(w)
        a[:, max(x - r, 0) : x + r, max(y - r, 0) : y + r] = 0
        return a


def dataloader(
    a,
    l=150,
    compose=Compose(
        [
            RandomCrop(32, 4),
            RandomHorizontalFlip(),
            ToTensor(),
            Cutout(4),
            Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ]
    ),
):
    return torch.utils.data.DataLoader(
        torchvision.datasets.ImageFolder(a, compose), 150, 1
    )


class DataLoaders:
    tfm = Compose(
        [
            RandomCrop(32, 4),
            RandomHorizontalFlip(),
            ToTensor(),
            # Cutout(4),
        ]
    )
    mean = torch.tensor((0.4914, 0.4822, 0.4465), device="cuda")[:, None, None]
    std = torch.tensor((0.2023, 0.1994, 0.2010), device="cuda")[:, None, None]

    def __init__(s, a, l=20, b=1, c=2):
        s.a = [glob(a + "/*") for a in glob(a + "/*")]
        s.l = l
        s.b = b
        s.c = c

    def __iter__(s):
        return s

    def __next__(s):
        a = [
            sample(
                [
                    (s.tfm(open(a)), i)
                    for i, a in enumerate(sample(s.a, s.c))
                    for a in a
                ],
                s.l,
            )
            for i in range(s.b)
        ]
        return (
            torch.stack([torch.stack([a[0] for a in a]) for a in a], 1)
            .cuda()
            .sub_(s.mean)
            .div_(s.std),
            torch.stack([torch.tensor([a[1] for a in a]) for a in a], 1).to(
                "cuda", torch.float
            ),
        )

In [ ]:
from pickle import load


class FastDataLoaders:
    def __init__(s, a="../dat/cifar-100-python/train", l=30, b=1):
        a = load(open(a, "rb"), encoding="bytes")
        s.a = (
            torch.tensor(
                a[b"data"][np.argsort(a[b"fine_labels"])].reshape(100, 500, 3, 32, 32),
                dtype=torch.float,
                device="cuda",
            )
            .sub_(
                torch.tensor((0.4914, 0.4822, 0.4465), device="cuda")[:, None, None]
                * 256
            )
            .div_(
                torch.tensor((0.2023, 0.1994, 0.2010), device="cuda")[:, None, None]
                * 256
            )
        )
        s.l = l
        s.b = b

    def __iter__(s):
        return s

    def __next__(s):
        x = torch.randint(1000, [s.l, s.b], device="cuda")
        y = x > 499
        return (
            s.a[
                torch.randint(100, [s.b, 2], device="cuda")[
                    torch.arange(s.b, device="cuda"), y.long()
                ],
                x % 500,
            ],
            y.float()[..., None],
        )

In [ ]:
def ConvBn(i, o=0, s=1, r=3, a=1):
    o = o or i
    return Sequential(
        Conv2d(i, o or i, r, s, 1, bias=0),
        BatchNorm2d(o or i, track_running_stats=0),
        *[Hardswish(1)] if a else [],
    )


class Res(Sequential):
    def forward(s, x):
        return super().forward(x) + x


def BaiduBlock(i, o):
    return Sequential(ConvBn(i, o), MaxPool2d(2), Res(ConvBn(o), ConvBn(o)))


def BaiduNet(o=16, k=64):
    return Sequential(
        ConvBn(3, k),
        BaiduBlock(k, k * 2),
        BaiduBlock(k * 2, k * 4),
        AdaptiveAvgPool2d(1),
        Flatten(),
        Linear(k * 4, o),
    )


class Add(Sequential):
    def forward(s, x):
        return s[0].forward(x) + s[1].forward(x)


def ResBlock(i, o=0, s=1):
    o = o or i
    return Sequential(
        Add(
            Sequential(ConvBn(i, o, s), ConvBn(o, a=0)),
            ConvBn(i, o, s, 1, 0) if i != o or s > 1 else Identity(),
        ),
        Hardswish(1),
    )


def ResNet(o=16, k=64, n=(1, 1, 1)):
    return Sequential(
        ConvBn(3, k),
        *(ResBlock(k) for _ in range(n[0])),
        ResBlock(k, k * 2, 2),
        *(ResBlock(k * 2) for _ in range(n[1] - 1)),
        ResBlock(k * 2, k * 4, 2),
        *(ResBlock(k * 4) for _ in range(n[2] - 1)),
        AdaptiveAvgPool2d(4),
        Flatten(),
        Linear(k * 16, o),
    )

In [ ]:
class BatchNorm(BatchNorm1d):
    def forward(s, x):
        return super().forward(x.view(-1, x.size(-1))).view(x.shape)


def unpack(a, n):
    if n < 9:
        return np.unpackbits(a.view("B")[::4, None], 1)[:, -n:]
    return np.unpackbits(a[:, None].view("B")[:, :2], 1)[
        :, [*range(8), *range(24 - n, 16)]
    ]


def los(x, y):
    return functional.binary_cross_entropy_with_logits(x, y.broadcast_to(x.shape))


def acc(x, y):
    x = x.signbit() != y
    return x.sum() / x.numel()


def fast_eval(m, d, x, y):
    i = torch.randint(len(d), d.shape[:1]) % torch.arange(1, len(d) + 1)
    return torch.cat((d, x[i]), 2), y[i]


def slow_eval(m, d, x, y):
    i, I = torch.tril_indices(*d.shape[:1] * 2)
    return m(torch.cat((d[i], x[I]), 2)), y[I]


def full_eval(m, d, x, y):
    return (
        m(
            torch.cat(
                (
                    d[:, None].broadcast_to(len(d), *d.shape),
                    x.broadcast_to(len(d), *x.shape),
                ),
                3,
            )
        ),
        y,
    )


def last_eval(m, d, x, y):
    w, W = m[0].weight.T.split((d.size(2), x.size(2)), 0)
    return m[1:](d @ w + x @ W), y

In [ ]:
cnn = BaiduNet(k=16).cuda()
rnn = GRU(17, 256).cuda()
mlp = Sequential(
    Linear(272, 256, 0),
    BatchNorm(256, track_running_stats=0),
    Hardswish(1),
    Linear(256, 1),
).cuda()
o = torch.optim.Adam((*cnn.parameters(), *rnn.parameters(), *mlp.parameters()))
s = torch.cuda.amp.GradScaler()

In [ ]:
for i, (x, y) in enumerate(FastDataLoaders(l=32, b=64)):
    o.zero_grad()
    with torch.cuda.amp.autocast():
        x = cnn(x.view(-1, 3, 32, 32)).view(-1, x.size(1), 16)
        x, y = full_eval(mlp, rnn(torch.cat((x, y), 2))[0], x, y)
        l = los(x, y)
    s.scale(l).backward()
    s.step(o)
    s.update()
    if i % 100 == 99:
        print(l, acc(x, y))